# NautilusTrader → canonical bundle → implementation-risk audit

The same toy strategy in **NautilusTrader 1.231.0** (Python 3.12+): a
low-level `BacktestEngine` with Nautilus's own `SimpleMovingAverage`
indicators, a NETTING/MARGIN venue (a cash account would deny the short) and a
`FixedFeeModel` of USD 1.00 per order. A market order submitted in `on_bar`
fills at that bar's close.

The exporter reads the four reports Nautilus generates after a run. This
notebook passes them to `export_nautilustrader_reports` itself, reading them
from `engine.trader`. (In 1.5.2 the convenience call
`export_nautilustrader_engine(engine, venue, ...)` looks for the report methods
on the engine instead of `engine.trader` and fails on a real engine; the fix
is on branch `wave3/adapter-e2e`.) Outputs below come from
`jupyter nbconvert --execute --to notebook --inplace 04_nautilustrader.ipynb`.

In [1]:
from pathlib import Path

import pandas as pd

import audit_kit as kit
import backtest_auditor
import toy_strategy as ts
import warnings

import nautilus_trader
from nautilus_trader.backtest.config import BacktestEngineConfig
from nautilus_trader.backtest.engine import BacktestEngine
from nautilus_trader.backtest.models import FillModel, FixedFeeModel
from nautilus_trader.config import LoggingConfig, StrategyConfig
from nautilus_trader.indicators import SimpleMovingAverage
from nautilus_trader.model.currencies import USD
from nautilus_trader.model.data import Bar, BarType
from nautilus_trader.model.enums import AccountType, OmsType, OrderSide, TimeInForce
from nautilus_trader.model.identifiers import InstrumentId, Symbol, TraderId, Venue
from nautilus_trader.model.instruments import Equity
from nautilus_trader.model.objects import Money, Price, Quantity
from nautilus_trader.trading.strategy import Strategy

from backtest_auditor.implementation_risk import (
    ComparisonLevel,
    build_v1_5d3_service,
    export_nautilustrader_reports,
)

pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 90)
REPO = Path.cwd().resolve().parents[1]
package_dir = Path(backtest_auditor.__file__).resolve().parent
print(
    "backtest_auditor",
    backtest_auditor.__version__,
    "from",
    package_dir.relative_to(REPO),
)
# nautilus_trader 1.231.0 calls a pandas API that pandas 3.1 deprecates; the
# warning comes from inside the engine and does not concern this example.
warnings.filterwarnings("ignore", message="Timestamp.utcnow is deprecated")
print("nautilus_trader", nautilus_trader.__version__)

backtest_auditor 1.6.0 from src/backtest_auditor
nautilus_trader 1.231.0


## 1. The data and the independent reference

`toy_strategy.py` holds the bars and computes the expected result with pandas
and `decimal` only (no engine, no auditor): ten fills, five round trips
including one short, final equity USD 99,844.00.

In [2]:
prices = ts.load_prices()
expected = ts.expected_run()
print(len(prices), "bars,", prices.index[0], "to", prices.index[-1])
prices.head(3)

400 bars, 2024-01-02 21:00:00+00:00 to 2025-07-14 20:00:00+00:00


,open,high,low,close,volume
timestamp,,,,,
2024-01-02 21:00:00+00:00,100.00,100.23,99.79,100.00,1000000.0
2024-01-03 21:00:00+00:00,100.19,101.07,100.06,100.99,1000000.0
2024-01-04 21:00:00+00:00,100.69,101.40,97.34,97.43,1000000.0


## 2. The strategy in NautilusTrader

In [3]:
VENUE = Venue("SIM")
INSTRUMENT_ID = InstrumentId(Symbol(ts.INSTRUMENT), VENUE)
BAR_TYPE = BarType.from_str(f"{INSTRUMENT_ID}-1-DAY-LAST-EXTERNAL")


class MovingAverageCrossConfig(StrategyConfig, frozen=True):
    lag: int = 1  # lag=0 is the look-ahead bug


class MovingAverageCross(Strategy):
    def __init__(self, config: MovingAverageCrossConfig) -> None:
        super().__init__(config)
        self.fast = SimpleMovingAverage(ts.FAST)
        self.slow = SimpleMovingAverage(ts.SLOW)
        self.lag = config.lag
        self.previous_signal = 0

    def on_start(self) -> None:
        self.register_indicator_for_bars(BAR_TYPE, self.fast)
        self.register_indicator_for_bars(BAR_TYPE, self.slow)
        self.subscribe_bars(BAR_TYPE)

    def on_bar(self, bar: Bar) -> None:
        ready = self.fast.initialized and self.slow.initialized
        long_signal = 1 if ready and self.fast.value > self.slow.value else 0
        session = (
            pd.Timestamp(bar.ts_event, unit="ns", tz="UTC")
            .tz_convert(ts.EXCHANGE_TIMEZONE)
            .normalize()
            .tz_localize(None)
        )
        in_short = (
            ts.SHORT_FIRST_SIGNAL_SESSION <= session <= ts.SHORT_LAST_SIGNAL_SESSION
        )
        signal = -1 if in_short else long_signal
        acting = signal if self.lag == 0 else self.previous_signal
        self.previous_signal = signal
        delta = acting * ts.QUANTITY - int(self.portfolio.net_position(INSTRUMENT_ID))
        if delta:
            self.submit_order(
                self.order_factory.market(
                    INSTRUMENT_ID,
                    OrderSide.BUY if delta > 0 else OrderSide.SELL,
                    Quantity.from_int(abs(delta)),
                    time_in_force=TimeInForce.GTC,
                )
            )


def bars() -> list[Bar]:
    return [
        Bar(
            BAR_TYPE,
            Price.from_str(f"{row.open:.2f}"),
            Price.from_str(f"{row.high:.2f}"),
            Price.from_str(f"{row.low:.2f}"),
            Price.from_str(f"{row.close:.2f}"),
            Quantity.from_int(int(row.volume)),
            timestamp.value,
            timestamp.value,
        )
        for timestamp, row in prices.iterrows()
    ]


def run_nautilus(lag: int) -> BacktestEngine:
    engine = BacktestEngine(
        config=BacktestEngineConfig(
            trader_id=TraderId("EXAMPLE-001"),
            logging=LoggingConfig(log_level="ERROR", bypass_logging=True),
        )
    )
    engine.add_venue(
        venue=VENUE,
        oms_type=OmsType.NETTING,
        account_type=AccountType.MARGIN,
        base_currency=USD,
        starting_balances=[Money(ts.INITIAL_CASH, USD)],
        fee_model=FixedFeeModel(Money(ts.FEE, USD)),
        fill_model=FillModel(prob_fill_on_limit=1.0, prob_slippage=0.0, random_seed=1),
    )
    engine.add_instrument(
        Equity(
            instrument_id=INSTRUMENT_ID,
            raw_symbol=Symbol(ts.INSTRUMENT),
            currency=USD,
            price_precision=2,
            price_increment=Price.from_str("0.01"),
            lot_size=Quantity.from_int(1),
            ts_event=0,
            ts_init=0,
        )
    )
    engine.add_data(bars())
    engine.add_strategy(MovingAverageCross(MovingAverageCrossConfig(lag=lag)))
    engine.run()
    return engine

## 3. Export, map to canonical bundle 1.0, check against the reference

The D3 adapter maps orders, fills, trades, positions, cash (the account
balance) and costs. It has no equity or return level, and positions only at
round-trip closes, so the audit below compares fills, trades, cash and
costs. The account is a MARGIN account, whose balance does not debit the
notional of an open position; the reference cash is therefore the matching
margin balance (`kit.margin_balances()`), built from the reference trades.

NautilusTrader puts a random UUID into every position id, so the trade ids,
`trades.csv` and the bundle's hashes change from run to run; the compared
values do not.

In [4]:
EXPORTS = kit.OUTPUT / "nautilustrader"


def export_run(engine: BacktestEngine, name: str):
    trader = engine.trader  # a BacktestEngine generates its reports here
    exported = export_nautilustrader_reports(
        trader.generate_orders_report(),
        trader.generate_fills_report(),
        trader.generate_positions_report(),
        trader.generate_account_report(VENUE),
        EXPORTS / f"export-{name}",
        run_id=f"nautilustrader-{name}",
        base_currency=ts.CURRENCY,
        overwrite=True,
    )
    return build_v1_5d3_service().adapt(exported)


engine = run_nautilus(lag=1)
adapted = export_run(engine, "correct")
engine.dispose()
print("adapter warnings:", [warning.code for warning in adapted.warnings])
bundle = kit.write_bundle(adapted.artifacts, EXPORTS / "nautilustrader-correct.zip")
kit.bundle_summary(EXPORTS / "nautilustrader-correct.zip")

adapter warnings: ['nautilustrader.position_fill_links_unavailable', 'nautilustrader.position_snapshots_quantity_only']


,file,records,sha256
0,cash_balances.csv,11,729ef3b8f5a2…
1,costs.csv,10,dff4968c8941…
2,fills.csv,10,ecd0b4c41e00…
3,orders.csv,10,43d9f399feee…
4,positions.csv,5,776059f18331…
5,trade_fills.csv,0,b65094a5fdf1…
6,trades.csv,5,e2ad85f7e5ec…


In [5]:
kit.compare_fills(bundle, expected)

,timestamp (UTC),engine,reference,same
0,2024-03-25 20:00,buy 100 @ 89.51,buy 100 @ 89.51,True
1,2024-06-19 20:00,sell 100 @ 97.33,sell 100 @ 97.33,True
2,2024-08-12 20:00,buy 100 @ 101.88,buy 100 @ 101.88,True
3,2024-09-25 20:00,sell 100 @ 94.88,sell 100 @ 94.88,True
4,2024-11-11 21:00,buy 100 @ 97.56,buy 100 @ 97.56,True
5,2025-01-24 21:00,sell 100 @ 94.96,sell 100 @ 94.96,True
6,2025-02-07 21:00,sell 100 @ 96.59,sell 100 @ 96.59,True
7,2025-03-07 21:00,buy 100 @ 91.56,buy 100 @ 91.56,True
8,2025-04-15 20:00,buy 100 @ 102.53,buy 100 @ 102.53,True
9,2025-06-03 20:00,sell 100 @ 97.82,sell 100 @ 97.82,True


In [6]:
kit.final_value_check(bundle, expected)

,engine,reference
,,
final cash (the run ends flat),99844,99844.00


## 4. Implementation-risk audit of the correct run

The reference bundle is built from `toy_strategy.py` alone. Levels are aligned
on economics (time, instrument, side), not on engine-local ids. Materiality
thresholds sit far above float noise and far below the effect of a one-bar
shift (see `audit_kit.MEASURED_FIELDS`). Levels compared for NautilusTrader:
fill, trade, cash, cost.

How to read the summary: a score near zero means every measured difference
is float noise below its threshold. No score with uncertainty `not_assessed`
means there was no numeric observation to classify, so read it with the
findings: with no `adverse` or `evidence_gap` finding it is exact agreement on
every measured field; with `evidence_gap` findings it means records could not
be paired at all (the NautilusTrader look-ahead run shows that case).

In [7]:
LEVELS = (
    ComparisonLevel.FILL,
    ComparisonLevel.TRADE,
    ComparisonLevel.CASH,
    ComparisonLevel.COST,
)
reference_bundle = kit.reference_artifacts(cash=kit.margin_balances())
correct = kit.implementation_risk(
    reference_bundle, bundle, LEVELS, name="nautilustrader-correct"
)
kit.summary(correct)

,implementation_risk_score,uncertainty,score_withheld_reason,supporting findings
candidate,,,,
nautilustrader-correct,None,not_assessed,None,1


In [8]:
kit.levels_table(correct)

,status,matched_different,matched_equal
level,,,
fill,compared,10,0
trade,compared,5,0
cash,compared,0,11
cost,compared,0,10


`matched_different` counts matched records in which any field differs,
including fields that only one side fills in (the reference leaves market
values and exposures empty) and labels that differ by design (a trade's
grouping method). Which fields differ, and whether the measurement policy
measures them:

In [9]:
kit.differing_fields(correct)

,level,field,records,measured
0,fill,liquidity_role,10,False
1,trade,grouping_method,5,False


In [10]:
print(kit.report_excerpt(correct))

# Backtest Implementation-Risk Report

- Report schema: `1.0`
- Reconciliation: `toy-ma-cross`
- Reference: `reference-pandas` (pandas-reference)
- Candidates: 1
- Overall status: not produced

> The report identifies implementation divergence against a declared reference baseline. It does not establish that any engine is correct, prove causation, certify strategy skill, or predict future performance.

## Policy identity

| Policy | SHA-256 |
|---|---|
| Normalization | `2519136a87b48b20220ac1a1a8ab296b0d13b98585b0f3ebf019bd142855c4cd` |
| Comparison | `42576e6f0725833ef11cacae1c12efd330d6c9b572a8580a339be6955daccd26` |
| Measurement | `304c2947c5538ed0607da46c18ae04035b58e2f00811ea78d519e14cf7f40a75` |
| Assessment | `74f6c652817ae763ea02898e9e78882780cb827132b8af0274a82d512a6d77ac` |

## Candidate `nautilustrader-correct`

- Engine: nautilustrader
- Implementation-risk score: withheld
- Uncertainty: not_assessed

### Comparison levels

| Level | Status | Alignment | Exact match | Rec

## 5. The same strategy with a one-bar look-ahead

The defect: the position follows the signal of the same bar, so the strategy
trades on a close it could not have known. Without an equity or return level, almost no Nautilus record exists at the same instant in both runs once every fill moves a bar (only the opening balance does), so the shift shows up as one-sided records (evidence gaps), not as a measured divergence.

In [11]:
engine = run_nautilus(lag=0)
lookahead_adapted = export_run(engine, "lookahead")
engine.dispose()
lookahead_bundle = kit.write_bundle(
    lookahead_adapted.artifacts, EXPORTS / "nautilustrader-lookahead.zip"
)
lookahead = kit.implementation_risk(
    reference_bundle, lookahead_bundle, LEVELS, name="nautilustrader-lookahead"
)
kit.summary(lookahead)

,implementation_risk_score,uncertainty,score_withheld_reason,evidence_gap findings
candidate,,,,
nautilustrader-lookahead,None,not_assessed,None,4


In [12]:
kit.levels_table(lookahead)

,status,candidate_only,reference_only,matched_equal
level,,,,
fill,compared,10,10,0
trade,compared,5,5,0
cash,compared,10,10,1
cost,compared,10,10,0


In [13]:
kit.findings_counts(lookahead)

count
kind         code                                                   
evidence_gap IMPLEMENTATION_FAMILY_EVIDENCE_IMPAIRED               3
             IMPLEMENTATION_ATTRIBUTION_INSUFFICIENT_EVIDENCE      1

## What this shows, and what it does not

- The NautilusTrader run and the independent reference agree on every measured
  field at every level compared above (other differences are confined to
  fields the reference leaves empty, labels that differ by design, or float
  noise below the thresholds), and the audit names no defect.
- The injected one-bar look-ahead is visible, but only as one-sided fills, costs, trades and
  balances (evidence gaps): nothing can be measured, so there is
  no score and no adverse finding. A per-bar equity record in the
  exporter would let the same shift surface as a measured
  divergence.
- The audit compares evidence against a declared reference under explicit
  thresholds. It does not prove either implementation correct, and it says
  nothing about whether the strategy is any good.
- The full JSON and Markdown reports of both runs are in `_output/` after
  executing this notebook.